# Error analysis
Recreate the fixed split, load the frozen model, and inspect false negatives/positives by amount and time. V components remain anonymous.

In [ ]:
import pandas as pd

from fraudshield.data import load_dataset, stratified_train_test_split
from fraudshield.persistence import load_model_bundle

frame = load_dataset("../data/raw/creditcard.csv")
_, x_test, _, y_test = stratified_train_test_split(frame)
model, metadata, threshold, metrics = load_model_bundle("../artifacts/model")
analysis = x_test.copy()
analysis["actual"] = y_test
analysis["score"] = model.predict_proba(x_test)[:, 1]
analysis["predicted"] = analysis.score >= threshold

In [ ]:
analysis[(analysis.actual == 1) & ~analysis.predicted].sort_values("Amount", ascending=False).head(
    20
)

In [ ]:
analysis.assign(amount_bucket=pd.qcut(analysis.Amount, 5, duplicates="drop")).groupby(
    "amount_bucket", observed=True
).agg(fraud=("actual", "sum"), reviews=("predicted", "sum"), mean_score=("score", "mean"))